# Cross-Entropy Loss

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 05.02 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/03_cross_entropy_loss.ipynb)

---

## 🎯 Learning Objective

Derive cross-entropy from entropy and KL divergence, and recognize it as THE loss function behind classification and language modeling.

---

## 📐 Theory

Every time you train a classifier or a language model, you are minimizing cross-entropy —
whether the code says `nn.CrossEntropyLoss()` or not. This notebook makes that fact completely
explicit, because "just trust the loss function" is not good enough once you're trying to
understand or debug a model.

### Definition

For a true distribution $P$ and a predicted distribution $Q$ over the same outcome space
$\mathcal{X}$, the **cross-entropy** of $Q$ relative to $P$ is

$$H(P, Q) = -\sum_{x\in\mathcal{X}} P(x) \log_2 Q(x) = \mathbb{E}_{x\sim P}\big[-\log_2 Q(x)\big]$$

Compare this to entropy from `05.01`, $H(P) = -\sum_x P(x)\log_2 P(x)$: the only difference is
that the *log* is evaluated at $Q$ while the *weighting* stays at $P$. Cross-entropy asks: if
outcomes are truly drawn from $P$, but I use $Q$'s probabilities to build my code lengths /
surprisal measure, how many bits do I pay *on average*? If $Q$ badly underestimates the
probability of something $P$ says happens often, that mismatch gets weighted heavily and the
cost balloons.

### The identity that ties it all together

Subtracting entropy from cross-entropy leaves exactly the mismatch term studied in `05.02`:

$$H(P,Q) - H(P) = -\sum_x P(x)\log_2 Q(x) + \sum_x P(x) \log_2 P(x) = \sum_x P(x) \log_2\frac{P(x)}{Q(x)} = D_{KL}(P\|Q)$$

Rearranged, this is the single most useful equation in this notebook:

$$\boxed{H(P, Q) = H(P) + D_{KL}(P\|Q)}$$

Cross-entropy decomposes into two pieces: $H(P)$, the *irreducible* cost from $P$'s own inherent
uncertainty (you'd pay this even with a perfect model, $Q=P$), and $D_{KL}(P\|Q)\ge 0$, the
*avoidable* extra cost specifically caused by $Q$ being wrong. Since $H(P)$ doesn't depend on
$Q$ at all, minimizing cross-entropy over $Q$ is *exactly the same optimization problem* as
minimizing $D_{KL}(P\|Q)$ over $Q$ — training with cross-entropy loss silently is training to
minimize KL divergence to the true data distribution. We verify this identity numerically below
by computing all three quantities independently and confirming they satisfy the equation to
numerical precision — not just asserting it.

### Cross-entropy as a loss function (log loss)

In supervised classification, $P$ is almost always a one-hot distribution: the true label $y$
gets probability 1, every other class gets 0 (or a slightly softened version under **label
smoothing**, which is precisely a deliberate move away from one-hot towards a higher-entropy
target — motivated by the max-entropy intuition from `05.01`). When $P$ is one-hot at class $y$,
the sum collapses to a single term:

$$H(P, Q) = -\sum_x P(x)\log_2 Q(x) = -\log_2 Q(y)$$

This is exactly **log loss**: penalize the model by the (negative log) probability it assigned
to the correct answer. Predicting $Q(y)$ close to 1 costs almost nothing; predicting $Q(y)$
close to 0 costs an enormous, unboundedly growing penalty — which is why cross-entropy loss
punishes *confident wrong answers* far more harshly than *hedged* ones. Note also that when $P$
is one-hot, $H(P) = 0$ exactly (a certain event has zero entropy — recall property 1 from
`05.01`), so the identity above simplifies to $H(P,Q) = D_{KL}(P\|Q)$: for hard-label
classification, cross-entropy loss and KL divergence to the true label are *literally the same
number*, not just related in spirit.

### Why every framework's loss uses natural log, not log base 2

`PyTorch`'s `nn.CrossEntropyLoss` and `F.cross_entropy` compute $-\ln Q(y)$ using natural log
(nats), not $-\log_2 Q(y)$ (bits) — purely a convention, changing log base only rescales the
loss by the constant factor $\ln 2 \approx 0.693$ and never changes which $Q$ minimizes it. This
notebook uses base-2 bits throughout for consistency with `05.01`–`05.02`, but we'll explicitly
convert and cross-check against `torch`'s nats-based computation below so the two conventions
are never silently conflated.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Sweeping a model's predicted distribution $Q$ while holding the true distribution $P$ fixed
shows cross-entropy bottoming out exactly at $Q=P$ — and its minimum value is exactly $H(P)$,
with everything above that floor being $D_{KL}(P\|Q)$.

In [ ]:
def entropy_bits(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log2(p))

def cross_entropy_bits(p, q, eps=1e-12):
    q = np.clip(q, eps, 1.0)
    return -np.sum(p * np.log2(q))

# Fix the true distribution P = (0.8, 0.2); sweep the model's predicted Q = (t, 1-t) over t
p_true = np.array([0.8, 0.2])
t_grid = np.linspace(0.01, 0.99, 300)
ce_values = np.array([cross_entropy_bits(p_true, np.array([t, 1 - t])) for t in t_grid])
kl_values = ce_values - entropy_bits(p_true)   # D_KL(P||Q) = H(P,Q) - H(P), the gap above the floor

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(t_grid, ce_values, color="#4C72B0", lw=2, label="cross-entropy H(P,Q)")
ax.axhline(entropy_bits(p_true), color="#55A868", ls="--", lw=1.5, label=f"H(P) = {entropy_bits(p_true):.4f} bits (floor)")
ax.axvline(0.8, color="gray", ls=":", lw=1)
ax.fill_between(t_grid, entropy_bits(p_true), ce_values, alpha=0.15, color="#DD8452",
                 label="D_KL(P||Q) = the avoidable gap")
ax.scatter([0.8], [entropy_bits(p_true)], color="#55A868", zorder=5, s=60)
ax.annotate("minimum at Q=P\nH(P,Q)=H(P), D_KL=0", (0.8, entropy_bits(p_true)),
            textcoords="offset points", xytext=(15, 15), fontsize=9)
ax.set_xlabel("Q = (t, 1-t)  -- model's predicted P(class 0) = t")
ax.set_ylabel("bits")
ax.set_title("True P=(0.8, 0.2) fixed; cross-entropy bottoms out exactly at Q=P")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

best_t = t_grid[np.argmin(ce_values)]
print(f"Cross-entropy minimized at t={best_t:.3f} (true P(class 0)=0.8) -- confirms minimizing")
print(f"H(P,Q) over Q recovers P exactly, which is why cross-entropy loss trains models TOWARD")
print(f"the true data distribution rather than some other target.")
print(f"Minimum cross-entropy value: {ce_values.min():.4f} bits  vs  H(P)={entropy_bits(p_true):.4f} bits")

## 🐍 Implementation from Scratch

We implement cross-entropy directly, then compute entropy, cross-entropy, and KL divergence
completely independently on the same example distributions and confirm
$H(P,Q) = H(P) + D_{KL}(P\|Q)$ holds numerically -- for both soft and one-hot true
distributions -- before cross-checking everything against `torch`'s built-in loss.

In [ ]:
from scipy.stats import entropy as scipy_entropy

def kl_divergence_bits(p, q, eps=1e-12):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    support = p > 0
    if np.any(q[support] == 0):
        return np.inf
    return np.sum(p[support] * np.log2(p[support] / q[support]))

# --- Independently compute H(P), H(P,Q), D_KL(P||Q) on the SAME soft-label example,
# then check the identity H(P,Q) = H(P) + D_KL(P||Q) numerically, not by construction ---
p_soft = np.array([0.7, 0.1, 0.1, 0.1])   # a "soft label" -- e.g. label smoothing or genuine ambiguity
q_pred = np.array([0.4, 0.3, 0.2, 0.1])   # model's predicted distribution, clearly different from p_soft

H_p = entropy_bits(p_soft)
H_pq = cross_entropy_bits(p_soft, q_pred)
D_kl = kl_divergence_bits(p_soft, q_pred)

print("Soft-label example: P =", p_soft, " Q =", q_pred)
print(f"  H(P)             = {H_p:.6f} bits   (scipy check: {scipy_entropy(p_soft, base=2):.6f})")
print(f"  H(P,Q)           = {H_pq:.6f} bits")
print(f"  D_KL(P||Q)       = {D_kl:.6f} bits   (scipy check: {scipy_entropy(p_soft, q_pred, base=2):.6f})")
print(f"  H(P) + D_KL(P||Q) = {H_p + D_kl:.6f} bits")
print(f"  IDENTITY HOLDS: {np.isclose(H_pq, H_p + D_kl)}")

# --- Same check for the actual classification case: P one-hot at the true class ---
y_true = 0
p_onehot = np.zeros(4); p_onehot[y_true] = 1.0
H_p_oh = entropy_bits(p_onehot)
H_pq_oh = cross_entropy_bits(p_onehot, q_pred)
D_kl_oh = kl_divergence_bits(p_onehot, q_pred)

print(f"\nOne-hot (hard label) example: P = {p_onehot}  Q = {q_pred}")
print(f"  H(P)       = {H_p_oh:.8f} bits  (should be exactly 0 -- a certain event has no entropy)")
print(f"  H(P,Q)     = {H_pq_oh:.6f} bits")
print(f"  D_KL(P||Q) = {D_kl_oh:.6f} bits")
print(f"  H(P,Q) == D_KL(P||Q) when P is one-hot: {np.isclose(H_pq_oh, D_kl_oh)}")
print(f"  -log2(Q[y_true]) directly: {-np.log2(q_pred[y_true]):.6f} bits  (matches H(P,Q) exactly: log loss)")

# --- Cross-check against torch's nn.CrossEntropyLoss, which uses NATURAL log and takes raw logits ---
logits = torch.log(torch.tensor(q_pred, dtype=torch.float64)).unsqueeze(0)  # reconstruct logits with softmax(logits)=q_pred
label = torch.tensor([y_true])
torch_ce_nats = torch.nn.functional.cross_entropy(logits, label).item()
our_ce_nats = H_pq_oh * np.log(2)   # bits -> nats: multiply by ln(2)

print(f"\ntorch.nn.functional.cross_entropy (nats) = {torch_ce_nats:.6f}")
print(f"our H(P,Q) converted bits->nats            = {our_ce_nats:.6f}")
print(f"match: {np.isclose(torch_ce_nats, our_ce_nats)}  "
      f"(confirms base-2 vs natural log are the same quantity, just rescaled by ln(2)={np.log(2):.4f})")

## 🤖 Why This Matters for AI

Cross-entropy is not "one of many" loss functions for classification and language modeling —
it is essentially *the* loss function. `torch.nn.CrossEntropyLoss` (classification heads),
the causal language modeling loss in GPT-style training (next-token prediction, where $P$ is a
one-hot at the true next token and $Q$ is the model's softmax output), and even the
reconstruction term of a VAE's ELBO when outputs are categorical, all reduce to the same
formula this notebook derives. Understanding $H(P,Q)=H(P)+D_{KL}(P\|Q)$ also demystifies why
loss never reaches exactly zero on real data: even a perfect model matching the true conditional
distribution ($Q=P$, so $D_{KL}=0$) still pays the irreducible $H(P)$ term — the entropy of
language itself. This is precisely the "entropy of natural text" constant that appears in the
Chinchilla scaling law, previewed here and derived fully in `05.07`.

In [ ]:
# Training a logistic regression classifier by gradient descent on cross-entropy (log) loss,
# and watching training loss converge toward the IRREDUCIBLE entropy floor H(Y|X) --
# the same H(P) term from this notebook's identity, now measured on a real training loop.
rng = np.random.default_rng(0)
true_w, true_b = 2.0, 0.5   # the (unknown, to the model) true data-generating parameters
n = 20_000
x = rng.normal(size=n)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def binary_entropy_bits(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

p_true_given_x = sigmoid(true_w * x + true_b)     # the TRUE P(y=1|x) -- inherently probabilistic, not deterministic
y = rng.binomial(1, p_true_given_x)                # observed noisy labels sampled from that true distribution

# The irreducible loss floor: even a PERFECT model (that recovers p_true_given_x exactly) still
# pays this average conditional entropy, because the labels themselves are genuinely stochastic
H_floor = np.mean(binary_entropy_bits(p_true_given_x))

# Train logistic regression via gradient descent, minimizing mean cross-entropy (bits) directly --
# this IS what torch.nn.BCEWithLogitsLoss / CrossEntropyLoss does under the hood
w, b, lr = 0.0, 0.0, 0.5
losses = []
for step in range(400):
    q = sigmoid(w * x + b)
    q_c = np.clip(q, 1e-12, 1 - 1e-12)
    losses.append(-np.mean(y * np.log2(q_c) + (1 - y) * np.log2(1 - q_c)))
    grad_z = (q - y) / np.log(2)      # d/dz [-(y log2 q + (1-y) log2(1-q))] = (q-y)/ln(2)
    w -= lr * np.mean(grad_z * x)
    b -= lr * np.mean(grad_z)

print(f"Learned (w, b) = ({w:.4f}, {b:.4f})   vs true (w, b) = ({true_w}, {true_b})")
print(f"\n{'step':>6} | {'loss (bits)':>12}")
for s in [0, 10, 50, 100, 200, 399]:
    print(f"{s:6d} | {losses[s]:12.4f}")
print(f"\nIrreducible floor H(Y|X) = {H_floor:.4f} bits  (population-level best ANY model could do)")
print(f"Final training loss      = {losses[-1]:.4f} bits")
print(f"Gap                      = {losses[-1] - H_floor:.4f} bits (near zero, up to finite-sample noise)")
print("The model converges almost exactly to the floor -- it has learned the TRUE conditional")
print("distribution as well as the noise in the labels allows. This is what")
print("H(P,Q) = H(P) + D_KL(P||Q) predicts: once D_KL(P||Q) -> 0, loss -> H(P). The tiny residual")
print("gap here (slightly above OR below zero) is finite-sample noise from evaluating on the same")
print("20,000 training points used to fit (w, b), not a violation of the population-level bound.")

## 🏋️ Exercises

### Warm-up
1. For a true one-hot label at class 2 (of 4 classes) and predicted $Q=(0.1, 0.2, 0.5, 0.2)$,
   compute $H(P,Q)$ by hand using $-\log_2 Q(y)$, then verify with `cross_entropy_bits` and
   confirm it equals `kl_divergence_bits(p_onehot, Q)`.

### Practice
2. Fix $P=(0.5, 0.3, 0.2)$ and sweep $Q$ over the 2-simplex (e.g. a grid of $(q_1,q_2)$ with
   $q_3=1-q_1-q_2$). Plot $H(P,Q)$ as a contour plot over the simplex and confirm the minimum
   sits exactly at $Q=P$ with value $H(P)$, generalizing this notebook's 2-class sweep to 3
   classes.

### Challenge
3. In the AI section, change `true_w` to a much larger value (e.g. `8.0`) so the true
   relationship between $x$ and $y$ becomes nearly deterministic almost everywhere. Recompute
   `H_floor` and retrain. Explain why the achievable training loss drops dramatically, and
   connect this to why some real-world prediction tasks (e.g. next-character prediction inside a
   quoted, memorized string) can achieve much lower loss than others (e.g. predicting the next
   word after "I think that...") — both are cross-entropy minimization, but against very
   different irreducible floors $H(P)$.

---

## 📚 Further Reading
- Shannon, ["A Mathematical Theory of Communication"](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf) (1948)
- Murphy, *Probabilistic Machine Learning: An Introduction*, Ch. 2 (Probability) and Ch. 10 (Logistic Regression)
- PyTorch documentation, [`torch.nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

---

*Next notebook: [Mutual Information](04_mutual_information.ipynb)*